# 🦀 Day 7: Mini-Project — REST API Bookmark Manager

---

## Project Overview

Build a **full CRUD REST API** for managing bookmarks:
- **GET** /bookmarks — list all
- **GET** /bookmarks/:id — get one
- **POST** /bookmarks — create
- **PUT** /bookmarks/:id — update
- **DELETE** /bookmarks/:id — delete

In-memory storage with `Arc<RwLock<HashMap<Uuid, Bookmark>>>`. This code is meant for a **Cargo binary** — EvCxR cannot run a long-lived server.

In [ ]:
:dep axum = "0.7"
:dep tokio = { version = "1", features = ["full"] }
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep uuid = { version = "1", features = ["v4"] }
:dep chrono = { version = "0.4", features = ["serde"] }

// Dependencies loaded.

## Data Model: Bookmark

```rust
struct Bookmark {
    id: Uuid,
    url: String,
    title: String,
    description: Option<String>,
    tags: Vec<String>,
    created_at: String,  // ISO 8601 for simplicity
}
```

In [ ]:
use serde::{Deserialize, Serialize};
use uuid::Uuid;

#[derive(Debug, Clone, Serialize, Deserialize)]
struct Bookmark {
    id: Uuid,
    url: String,
    title: String,
    description: Option<String>,
    tags: Vec<String>,
    created_at: String,
}

#[derive(Debug, Deserialize)]
struct CreateBookmark {
    url: String,
    title: String,
    description: Option<String>,
    tags: Option<Vec<String>>,
}

#[derive(Debug, Deserialize, Default)]
struct UpdateBookmark {
    url: Option<String>,
    title: Option<String>,
    description: Option<String>,
    tags: Option<Vec<String>>,
}

fn new_bookmark(create: CreateBookmark) -> Bookmark {
    Bookmark {
        id: Uuid::new_v4(),
        url: create.url,
        title: create.title,
        description: create.description,
        tags: create.tags.unwrap_or_default(),
        created_at: chrono::Utc::now().to_rfc3339(),
    }
}

let b = new_bookmark(CreateBookmark {
    url: "https://rust-lang.org".into(),
    title: "Rust".into(),
    description: Some("The Rust language".into()),
    tags: Some(vec!["rust".into(), "programming".into()]),
});
println!("Bookmark: {:?}", b);

## In-Memory Storage: Arc&lt;RwLock&lt;HashMap&gt;&gt;

Shared state across handlers. `RwLock` allows multiple readers or one writer.

In [ ]:
use std::collections::HashMap;
use std::sync::{Arc, RwLock};
use uuid::Uuid;

type BookmarkStore = Arc<RwLock<HashMap<Uuid, Bookmark>>>;

fn create_store() -> BookmarkStore {
    Arc::new(RwLock::new(HashMap::new()))
}

let store = create_store();
{
    let mut guard = store.write().unwrap();
    let b = Bookmark {
        id: Uuid::new_v4(),
        url: "https://example.com".into(),
        title: "Example".into(),
        description: None,
        tags: vec![],
        created_at: "".into(),
    };
    guard.insert(b.id, b);
}
let count = store.read().unwrap().len();
println!("Store has {} bookmark(s)", count);

## Routes and Handlers Overview

| Method | Route | Handler |
|--------|-------|----------|
| GET | /bookmarks | list_bookmarks |
| GET | /bookmarks/:id | get_bookmark |
| POST | /bookmarks | create_bookmark |
| PUT | /bookmarks/:id | update_bookmark |
| DELETE | /bookmarks/:id | delete_bookmark |

## Full Handler Implementations

Complete code for a Cargo project. Copy into `src/main.rs` and add `chrono` to Cargo.toml for real timestamps.

In [ ]:
// === Full Cargo project code — run with: cargo run ===
//
// [dependencies]
// axum = "0.7"
// tokio = { version = "1", features = ["full"] }
// serde = { version = "1", features = ["derive"] }
// serde_json = "1"
// uuid = { version = "1", features = ["v4"] }
// chrono = { version = "0.4", features = ["serde"] }
//
// use axum::{
//     Router, extract::{State, Path, Json}, routing::{get, post, put, delete},
//     http::StatusCode,
// };
// use serde::{Deserialize, Serialize};
// use std::collections::HashMap;
// use std::sync::{Arc, RwLock};
// use uuid::Uuid;
//
// #[derive(Clone)]
// struct AppState { store: BookmarkStore }
// type BookmarkStore = Arc<RwLock<HashMap<Uuid, Bookmark>>>;
//
// async fn list_bookmarks(State(state): State<AppState>) -> Json<Vec<Bookmark>> {
//     let guard = state.store.read().unwrap();
//     Json(guard.values().cloned().collect())
// }
//
// async fn get_bookmark(State(state): State<AppState>, Path(id): Path<Uuid>) -> Result<Json<Bookmark>, StatusCode> {
//     let guard = state.store.read().unwrap();
//     guard.get(&id).cloned().map(Json).ok_or(StatusCode::NOT_FOUND)
// }
//
// async fn create_bookmark(State(state): State<AppState>, Json(body): Json<CreateBookmark>) -> (StatusCode, Json<Bookmark>) {
//     let b = new_bookmark(body);
//     state.store.write().unwrap().insert(b.id, b.clone());
//     (StatusCode::CREATED, Json(b))
// }
//
// async fn update_bookmark(State(state): State<AppState>, Path(id): Path<Uuid>, Json(body): Json<UpdateBookmark>) -> Result<Json<Bookmark>, StatusCode> {
//     let mut guard = state.store.write().unwrap();
//     let b = guard.get_mut(&id).ok_or(StatusCode::NOT_FOUND)?;
//     if let Some(url) = body.url { b.url = url; }
//     if let Some(title) = body.title { b.title = title; }
//     if let Some(desc) = body.description { b.description = Some(desc); }
//     if let Some(tags) = body.tags { b.tags = tags; }
//     Ok(Json(b.clone()))
// }
//
// async fn delete_bookmark(State(state): State<AppState>, Path(id): Path<Uuid>) -> StatusCode {
//     if state.store.write().unwrap().remove(&id).is_some() {
//         StatusCode::NO_CONTENT
//     } else {
//         StatusCode::NOT_FOUND
//     }
// }
//
// #[tokio::main]
// async fn main() {
//     let state = AppState { store: create_store() };
//     let app = Router::new()
//         .route("/bookmarks", get(list_bookmarks).post(create_bookmark))
//         .route("/bookmarks/:id", get(get_bookmark).put(update_bookmark).delete(delete_bookmark))
//         .with_state(state);
//     let listener = tokio::net::TcpListener::bind("127.0.0.1:3000").await.unwrap();
//     axum::serve(listener, app).await.unwrap();
// }

println!("See comments for full Cargo project code.");

## Error Handling with HTTP Status Codes

| Case | Status |
|------|--------|
| Success (create) | 201 Created |
| Success (delete) | 204 No Content |
| Not found | 404 Not Found |
| Bad request (invalid JSON) | 400 Bad Request |

In [ ]:
use axum::{Router, extract::{State, Path}, Json, http::StatusCode, routing::get};

// Path params for UUID: use a wrapper or serde with Uuid
// Axum 0.7: Path<Uuid> works if Uuid implements FromStr
// uuid::Uuid does implement FromStr

async fn get_one(Path(id): Path<uuid::Uuid>) -> Result<String, StatusCode> {
    if id == uuid::Uuid::nil() {
        return Err(StatusCode::BAD_REQUEST);
    }
    Ok(format!("ID: {}", id))
}

let app = Router::new().route("/bookmarks/:id", get(get_one));
println!("Path<Uuid> for UUID path params; return Result for 404/400.");

## 🏋️ Exercise: Search/Filter or Pagination

Extend the bookmark API with one of:

1. **Search by tag**: `GET /bookmarks?tag=rust` — return only bookmarks containing that tag
2. **Pagination**: `GET /bookmarks?page=1&limit=10` — return a page of results

Implement the query struct, update `list_bookmarks` to filter/paginate, and return appropriate JSON.

In [ ]:
// YOUR CODE HERE
//
// Option A — Tag filter:
// #[derive(Deserialize)]
// struct BookmarkQuery { tag: Option<String> }
// In list_bookmarks: Query(q): Query<BookmarkQuery>
// Filter: bookmarks.iter().filter(|b| q.tag.as_ref().map_or(true, |t| b.tags.contains(t)))
//
// Option B — Pagination:
// #[derive(Deserialize)]
// struct Pagination { page: Option<u32>, limit: Option<u32> }
// Use .skip(offset).take(limit) on the iterator
// Return { items: Vec<Bookmark>, total: usize, page: u32 }

## 🎯 Key Takeaways

1. **Full CRUD REST API**: GET list, GET one, POST, PUT, DELETE
2. **Data model**: Bookmark with id (Uuid), url, title, description, tags, created_at
3. **Storage**: `Arc<RwLock<HashMap<Uuid, Bookmark>>>` for in-memory shared state
4. **DTOs**: CreateBookmark, UpdateBookmark for request bodies
5. **Error handling**: 201 Created, 204 No Content, 404 Not Found, 400 Bad Request
6. Run as a Cargo binary — EvCxR cannot host a long-lived server

---

**Week 15 Complete!** Next week: Databases & Deployment 🚀